In [1]:
import os
import sys
import time
import psutil
from glob import glob
from functools import partial
from shutil import copy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm

import pyscf
from pyscf import gto, scf, mcscf, cc
import ffsim

from qiskit import QuantumCircuit, QuantumRegister
from qiskit.primitives import StatevectorSampler, BitArray
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

from qiskit_addon_sqd.fermion import SCIResult, diagonalize_fermionic_hamiltonian
from qiskit_addon_sqd.counts import bit_array_to_arrays

from ansatzmap import get_zigzag_physical_layout
from DDLUCJ import DDLUCJ, GrabAmps



# 1. Try SLURM_NTASKS (multi-node / MPI tasks) or SLURM_CPUS_PER_TASK (multithreading)
slurm_tasks = os.environ.get("SLURM_NTASKS")
slurm_cpus_per_task = os.environ.get("SLURM_CPUS_PER_TASK")

if slurm_tasks or slurm_cpus_per_task:
    # Use SLURM allocation
    n_jobs = int(slurm_tasks or 1) * int(slurm_cpus_per_task or 1)
    env_type = "SLURM Cluster"
else:
    # Fallback for local Debian desktop (logical cores/threads)
    n_jobs = os.cpu_count()  # Or use psutil.cpu_count(logical=False) for physical cores only
    env_type = "Local Desktop"

print(f"[{env_type}] Configured n_jobs / threads: {n_jobs}")

[Local Desktop] Configured n_jobs / threads: 16


In [2]:
# # Create directories to store energies and output files
# os.makedirs('./energies', exist_ok=True)
# os.makedirs('./jobids', exist_ok=True)

# Load metadata
moldf = pd.read_csv('molecules.csv')
activespacedf = pd.read_csv("active_spaces.csv")
energyDF = pd.read_csv("../../../classical/energies.csv", index_col=0)

BasisSets = ['STO-3G', 'cc-pVDZ', 'aug-cc-pVDZ']

In [ ]:
results_data = []
for shots in np.logspace(4,7,3).astype(int):
    for row in tqdm(moldf.itertuples(), total=len(moldf), desc="Molecules"):
        moldict = row._asdict()
        name = moldict['molecule']
        n_electrons = int(moldict['n_electrons'])
        num_orbitals = int(moldict['num_orbitals'])
        xyzname = moldict['mol_filename']
        pathxyz = os.path.join("../../../classical/structures/", xyzname)
        
        for basis in BasisSets:
            ampdict = GrabAmps(name, basis)
            
            for k, v in ampdict.items():
                t1, t2 = v
                for L in range(1, 6):
                    energy_file = f"./energies/{name}_LUCJ_L{L}_{basis}_{k}_StateVector_Shots{shots}.txt"
                    
                    # Skip if already computed
                    # if os.path.exists(energy_file):
                    #     print(f"Skipping already calculated: {name}_L{L}_{basis}_{k}")
                    #     continue
    
                    print(f"--- Running Statevector + Fulqrum Postprocessing: {name} (L={L}, Basis={basis}, Amps={k}) ---")
                    
                    # 1. Initialize with local Statevector sampler mode
                    initDDLUCJ = DDLUCJ(
                        StructurePath=pathxyz, 
                        BasisSet=basis, 
                        NElec=n_electrons,
                        NOrb=num_orbitals,
                        injected=True,
                        t1=t1, 
                        t2=t2,
                        n_reps=L,
                        backend="statevector",
                        optimization_level=3,
                        shots=shots,
                        temp_dir="./",
                        clean_temp_dir=True,
                        n_jobs=1,  # Set n_jobs to 1 to prevent Dice C++ fallback
                        num_batches=10,
                        max_iterations=5,
                        samples_per_batch=1000,
                        verbose=True
                    )
                    
                    # 2. Execute sampling and Fulqrum postprocessing in one call
                    # Fulqrum directly returns (total_energy, subspace_dimension)
                    total_energy, subspace_dim = initDDLUCJ(
                        postprocess=True, 
                        usefulqrum=True
                    )
                    
                    # Convert scalar NumPy array/value to a standard Python float
                    energy_val = float(np.squeeze(total_energy))
                    
                    print(f"Calculated Total Energy: {energy_val:.8f} Ha | Subspace Dim: {subspace_dim}")
                    
                    # 3. Save results to text file
                    with open(energy_file, 'w') as f:
                        f.write(f"Basis Set: {basis}\n")
                        f.write(f"Molecule: {name}\n")
                        f.write(f"Method: LUCJ(L={L})/{k}\n")
                        f.write(f"Energy: {energy_val}\n")
                        f.write(f"Subspace Dimension: {subspace_dim}\n")
                    
                    results_data.append({
                        "Molecule": name,
                        "Basis Set": basis,
                        "Layers": L,
                        "Injection": k,
                        "Energy": energy_val,
                        "Subspace Dim": subspace_dim
                    })
                    
# Summary DataFrame
df_results = pd.DataFrame(results_data)
df_results.to_csv("statevector_fulqrum_energies.csv", index=False)
df_results.head()

Molecules:   0%|          | 0/12 [00:00<?, ?it/s]

--- Running Statevector + Fulqrum Postprocessing: ammonia (L=1, Basis=STO-3G, Amps=MP2) ---
Active Space Orbitals: 8, Electrons: 10, Frozen: 0
Active indices: [0, 1, 2, 3, 4, 5, 6, 7]
Transpiled circuit for local StatevectorSampler simulation.
Executing simulation using local StatevectorSampler...
ITERATION 0
  num total half strs:  5
  num selected half strs:  5
  Half strs construction took: 0.0000 seconds
  Subspace construction took: 0.000020 seconds
  Subspace dimension: 5 x 5 = 25
  Operator projection took: 0.006184 seconds
  CSR matrix memory: 0.002628 MBs
  Initial guess vector v0 construction took: 0.0000 seconds
  Starting eigensolving ...
  Eigensolving took: 0.0003 seconds
  Electronic Energy: [-67.4119156]
  Total Energy: [-55.46723785]
  num carryover full strs: 22
  num total half strs:  5
  num selected half strs:  5
  Half strs construction took: 0.0000 seconds
  Subspace construction took: 0.000008 seconds
  Subspace dimension: 5 x 5 = 25
  Operator projection took: 

In [ ]:
refdf = pd.read_csv("../../../classical/energies.csv",index_col=0).reset_index(drop=True)

In [ ]:
refdf.loc[(refdf['Name'] == 'ammonia')&(refdf['Basis Set'] == 'STO-3G')]